In [1]:
import os

In [2]:
from dotenv import load_dotenv

In [3]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [4]:
load_dotenv()

True

/////INDEXING (DOCUMENT INGESTION)

In [5]:
"""Fetches the transcript of a YouTube video."""
video_id = os.getenv("YOUTUBE_VIDEO_ID")
if not video_id:
    raise ValueError("YOUTUBE_VIDEO_ID environment variable is missing. Please check your .env file.")
try:
    
     ytt_api = YouTubeTranscriptApi()
     transcript_list = ytt_api.fetch(video_id, languages=["en"])
    
     transcript = " ".join(chunk.text for chunk in transcript_list)
     print(transcript)
    
except TranscriptsDisabled:
    print("No captions available for this video.")

Atropic CEO saying that more than half of the entry- level jobs will disappear. We've never had a technology that's this disruptive. It's bigger and it's broader and it's moving faster than anything has before. >> None of us should be scared of AI. One just has to figure out how to work with it. Then you're riding the title. People today, the entire world is buzzing with chatter about AI agents. But very few people are actually teaching you how to build them and how to use them for large enterprises. And most of the information that you have is coming from reals where somehow people present magical outputs using simple prompts. But that is actually very very shallow. So today we have Kedar who has had an amazing run at Amazon India and now heads marketing for Kotak Mahindra Bank and he's here to teach us what exactly is a multi-agent system, how to build a multi- aent AI system and how to take it step by step towards execution without any flaws. So if you've been watching reals on AI, 

//////INDEXING (TEXT SPLITTING)

In [6]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [7]:
print(chunks)

[Document(metadata={}, page_content="Atropic CEO saying that more than half of the entry- level jobs will disappear. We've never had a technology that's this disruptive. It's bigger and it's broader and it's moving faster than anything has before. >> None of us should be scared of AI. One just has to figure out how to work with it. Then you're riding the title. People today, the entire world is buzzing with chatter about AI agents. But very few people are actually teaching you how to build them and how to use them for large enterprises. And most of the information that you have is coming from reals where somehow people present magical outputs using simple prompts. But that is actually very very shallow. So today we have Kedar who has had an amazing run at Amazon India and now heads marketing for Kotak Mahindra Bank and he's here to teach us what exactly is a multi-agent system, how to build a multi- aent AI system and how to take it step by step towards execution without any flaws. So 

In [8]:
print(chunks[0])

page_content='Atropic CEO saying that more than half of the entry- level jobs will disappear. We've never had a technology that's this disruptive. It's bigger and it's broader and it's moving faster than anything has before. >> None of us should be scared of AI. One just has to figure out how to work with it. Then you're riding the title. People today, the entire world is buzzing with chatter about AI agents. But very few people are actually teaching you how to build them and how to use them for large enterprises. And most of the information that you have is coming from reals where somehow people present magical outputs using simple prompts. But that is actually very very shallow. So today we have Kedar who has had an amazing run at Amazon India and now heads marketing for Kotak Mahindra Bank and he's here to teach us what exactly is a multi-agent system, how to build a multi- aent AI system and how to take it step by step towards execution without any flaws. So if you've been watching

//// EMBEDDING GENERATION & STORING IN VECTORS

In [9]:
embedding=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=FAISS.from_documents(chunks,embedding)

In [10]:
vector_store.docstore._dict

{'1bdf20ab-01f4-457f-934d-d20b6fc6a5e3': Document(id='1bdf20ab-01f4-457f-934d-d20b6fc6a5e3', metadata={}, page_content="Atropic CEO saying that more than half of the entry- level jobs will disappear. We've never had a technology that's this disruptive. It's bigger and it's broader and it's moving faster than anything has before. >> None of us should be scared of AI. One just has to figure out how to work with it. Then you're riding the title. People today, the entire world is buzzing with chatter about AI agents. But very few people are actually teaching you how to build them and how to use them for large enterprises. And most of the information that you have is coming from reals where somehow people present magical outputs using simple prompts. But that is actually very very shallow. So today we have Kedar who has had an amazing run at Amazon India and now heads marketing for Kotak Mahindra Bank and he's here to teach us what exactly is a multi-agent system, how to build a multi- aent

In [11]:
result = vector_store.get_by_ids(['af34bb98-85f1-4f7c-9f11-df3aa7dc42e0'])
print(result)   

[]


/////RETRIEVAL

In [12]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke('What are ai agents?')

[Document(id='b3c3513b-8c93-415f-bf63-92fc3b1fbfca', metadata={}, page_content="fixed output. AI agent, it's like a coach. You're letting your team players perform the way they want, but you still want them playing in certain contours. >> How should I build an effective AI agent all the way to deployment? >> AI system is being built uh on three vectors. Let's call it the traffic d. Cost is what is most pertinent. Time is important. And the third is quality. I think the the mistake is building. Can you show me a demo of how an AI agent helps you achieve a certain outcome with minimal instructions? >> Solitaire credit card benefits. >> So that's it. That's your brief like a four-word brief. >> Frequent traveler unlock a complimentary flight after spending 1.5 lakhs. Apply now. >> This agent packed with all the capability should be able to do the job of this also. Hi Kar, welcome to the Indian business podcast. Kdar, something absolutely crazy is happening in the agentic AI space and it i

/////STEP:3 AUGMENTATION

In [13]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

prompt = PromptTemplate(
    template="""
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}
    Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [14]:
question       = "were the principles of making an effective agent discussed? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [15]:
print(retrieved_docs)

[Document(id='9705debd-1427-4b5b-a27a-e73ad9fa7c3e', metadata={}, page_content="agent to give you creative which will achieve high CTR. Is this the structure of a prompt? >> Uh at the end of the day, agents are just prompts, right? It's just a compilation of of a a well-defined prompt which then has probably as necessitates as as necessary will have API connections to to be done and it may have memory which is given from outside the world but the rest is all within within the prompt that that you expect it to work. It's just the way you how sharply you're able to define the the action it should take and the kind of reasoning that you want it to do. As long as that is clear, you'll perhaps get a lot more uh probable output that you know will work for the goal that you've put in. People after the board, I tried to go deep into what Kdar said and I found this research paper called the lost in the middle. In simple words, when people build agentic systems, the first instinct is to create o

In [16]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [17]:
context_text

"agent to give you creative which will achieve high CTR. Is this the structure of a prompt? >> Uh at the end of the day, agents are just prompts, right? It's just a compilation of of a a well-defined prompt which then has probably as necessitates as as necessary will have API connections to to be done and it may have memory which is given from outside the world but the rest is all within within the prompt that that you expect it to work. It's just the way you how sharply you're able to define the the action it should take and the kind of reasoning that you want it to do. As long as that is clear, you'll perhaps get a lot more uh probable output that you know will work for the goal that you've put in. People after the board, I tried to go deep into what Kdar said and I found this research paper called the lost in the middle. In simple words, when people build agentic systems, the first instinct is to create one super agent and dump a huge list of instructions on it. But that's exactly\n

In [18]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [19]:
final_prompt 

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n    If the context is insufficient, just say you don't know.\n\n    agent to give you creative which will achieve high CTR. Is this the structure of a prompt? >> Uh at the end of the day, agents are just prompts, right? It's just a compilation of of a a well-defined prompt which then has probably as necessitates as as necessary will have API connections to to be done and it may have memory which is given from outside the world but the rest is all within within the prompt that that you expect it to work. It's just the way you how sharply you're able to define the the action it should take and the kind of reasoning that you want it to do. As long as that is clear, you'll perhaps get a lot more uh probable output that you know will work for the goal that you've put in. People after the board, I tried to go deep into what Kdar said and I found this research paper called the lo

In [20]:
retrieved_docs

[Document(id='9705debd-1427-4b5b-a27a-e73ad9fa7c3e', metadata={}, page_content="agent to give you creative which will achieve high CTR. Is this the structure of a prompt? >> Uh at the end of the day, agents are just prompts, right? It's just a compilation of of a a well-defined prompt which then has probably as necessitates as as necessary will have API connections to to be done and it may have memory which is given from outside the world but the rest is all within within the prompt that that you expect it to work. It's just the way you how sharply you're able to define the the action it should take and the kind of reasoning that you want it to do. As long as that is clear, you'll perhaps get a lot more uh probable output that you know will work for the goal that you've put in. People after the board, I tried to go deep into what Kdar said and I found this research paper called the lost in the middle. In simple words, when people build agentic systems, the first instinct is to create o

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"agent to give you creative which will achieve high CTR. Is this the structure of a prompt? >> Uh at the end of the day, agents are just prompts, right? It's just a compilation of of a a well-defined prompt which then has probably as necessitates as as necessary will have API connections to to be done and it may have memory which is given from outside the world but the rest is all within within the prompt that that you expect it to work. It's just the way you how sharply you're able to define the the action it should take and the kind of reasoning that you want it to do. As long as that is clear, you'll perhaps get a lot more uh probable output that you know will work for the goal that you've put in. People after the board, I tried to go deep into what Kdar said and I found this research paper called the lost in the middle. In simple words, when people build agentic systems, the first instinct is to create one super agent and dump a huge list of instructions on it. But that's exactly\n

In [22]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [23]:
final_prompt

StringPromptValue(text="\n    You are a helpful assistant.\n    Answer ONLY from the provided transcript context.\n    If the context is insufficient, just say you don't know.\n\n    agent to give you creative which will achieve high CTR. Is this the structure of a prompt? >> Uh at the end of the day, agents are just prompts, right? It's just a compilation of of a a well-defined prompt which then has probably as necessitates as as necessary will have API connections to to be done and it may have memory which is given from outside the world but the rest is all within within the prompt that that you expect it to work. It's just the way you how sharply you're able to define the the action it should take and the kind of reasoning that you want it to do. As long as that is clear, you'll perhaps get a lot more uh probable output that you know will work for the goal that you've put in. People after the board, I tried to go deep into what Kdar said and I found this research paper called the lo

////////////STEP 4: GENERATION

In [24]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the principles of making an effective agent were discussed. The three key parameters mentioned are:

1. **Act** - The agent should be able to perform actions.
2. **Reason** - The agent should be capable of reasoning.
3. **Memory** - The agent needs to have memory to retain information.

These aspects are essential for the agent to achieve its goals effectively.


/////////////////BUILDING A CHAIN

In [25]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [26]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [27]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [28]:
parallel_chain.invoke('What is CTR?') 

{'context': "whether this content piece will generate enough viewership or not and if it doesn't cross a certain threshold then maybe the loop can start all over again where it will then ask the topic scouter to find another title and then carry out the entire process. >> Yes. Now this is exactly the way it behaves for us when we are making creatives. It comes with a score to tell us whether this creative is going to give you the right the CTRs the historical CTRs is going to be better or not. So it's a it's a it's more of a probabilistic output that comes along with the creative output that it has. So it will reduce the errors that one would have otherwise had. So to your point, yes, you could always have a prediction come to you on whether this episode is going to be viewed or not, but it's always going to be built on the past. Guys, it's simplify. You see, when you build an agent, you have to keep three important things in mind. Let's say you are building a customer support agent wi

In [29]:
parser = StrOutputParser()

In [30]:
main_chain = parallel_chain | prompt | llm | parser

In [31]:
main_chain.invoke('Can you summarize the video')

'The video discusses the use of AI agents in digital marketing, particularly for creating campaigns for credit cards. It highlights how a vague brief can be sent to an agent, which then generates a static campaign or video tailored to specific consumer segments. The process of creating videos, which traditionally takes a long time, is streamlined by the AI agent, allowing for quick production of personalized content. The speaker emphasizes the difference between automation and AI agents, illustrating how agents can adapt and deliver creative solutions based on defined KPIs. The session promises further exploration of multi-agent systems and their potential in building effective sales engines.'

//////////////EVALUATION METRICS

In [34]:
import json
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd
from datasets import Dataset
from langchain_core.prompts import PromptTemplate
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    context_precision,
)

# Aapke evaluator models pehle se defined hone chahiye (e.g., Langchain wrappers)
# Example:
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# evaluator_llm = ChatOpenAI(model="gpt-4o")
# evaluator_embeddings = OpenAIEmbeddings()

print("1. Generating dynamic test questions from the transcript...")

# Prompt for generating Q&A pairs
qa_gen_prompt = PromptTemplate(
    template="""
    You are an expert evaluator. Read the following video transcript and generate 4 factual questions and their correct answers based ONLY on the text.
    Return the output STRICTLY as a JSON format exactly like this, with no extra text or markdown:
    {{
        "qa_pairs": [
            {{"question": "Question 1", "ground_truth": "Answer 1"}},
            {{"question": "Question 2", "ground_truth": "Answer 2"}}
        ]
    }}

    Transcript:
    {transcript}
    """,
    input_variables=["transcript"]
)

# Generate JSON (Make sure 'llm' and 'parser' are defined in your environment)
qa_chain = qa_gen_prompt | llm | parser
raw_qa_output = qa_chain.invoke({"transcript": transcript})

# Parse JSON safely
clean_json = raw_qa_output.replace("```json", "").replace("```", "").strip()
generated_qa = json.loads(clean_json)

# Create variables for evaluation
dynamic_questions = [item["question"] for item in generated_qa["qa_pairs"]]
dynamic_ground_truths = [item["ground_truth"] for item in generated_qa["qa_pairs"]]

print("Questions Generated Successfully!")
for q in dynamic_questions:
    print(f"- {q}")

# ---------------------------------------------------------
print("\n2. Running your RAG pipeline to get answers and contexts...")
contexts = []
answers = []

for query in dynamic_questions:
    # Get the generated answer using your existing chain
    response = main_chain.invoke(query)
    answers.append(response)
    
    # Get the raw retrieved documents using your existing retriever
    retrieved_docs = retriever.invoke(query)
    contexts.append([doc.page_content for doc in retrieved_docs])

# Format into a HuggingFace Dataset
data = {
    "user_input": dynamic_questions,         
    "response": answers,                     
    "retrieved_contexts": contexts,          
    "reference": dynamic_ground_truths       
}
evaluation_dataset = Dataset.from_dict(data)

# ---------------------------------------------------------
print("\n3. Running RAGAS Evaluation (This checks Faithfulness, Relevancy, Precision, and Recall)...")

# FIX: evaluate() function mein llm aur embeddings explicitly pass karein.
# Isse NaN ka issue solve ho jayega kyunki metrics ab properly embed kar payenge.
evaluator_llm = ChatOpenAI(model="gpt-4o") # Ya gpt-3.5-turbo use kar sakte hain
evaluator_embeddings = OpenAIEmbeddings()
result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=evaluator_llm,               # <-- NEW: Explicitly passing evaluator LLM
    embeddings=evaluator_embeddings  # <-- NEW: Explicitly passing evaluator Embeddings
)

print("\n4. Evaluation Complete! Here is your accuracy report:")
# Convert to a Pandas table and display
df_results = result.to_pandas()
display(df_results)

# Calculate overall average score
average_score = df_results[['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']].mean().mean()
print(f"\nOverall Pipeline Accuracy Score: {average_score:.2f} out of 1.0")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_23052\1034744593.py:8: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\ASUS\AppData\Local\Temp\ipykernel_23052\1034744593.py:8: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\ASUS\AppData\Local\Temp\ipykernel_23052\1034744593.py:8: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import (
C:\Users\ASUS\AppData\Local\Temp\ipykernel_23052\103474

1. Generating dynamic test questions from the transcript...
Questions Generated Successfully!
- What does the CEO of Atropic say about entry-level jobs?
- What is the main focus of the podcast discussed in the transcript?
- What are the three vectors on which an AI system is built according to Kedar?
- What is the difference between automation and AI agents as explained in the transcript?

2. Running your RAG pipeline to get answers and contexts...

3. Running RAGAS Evaluation (This checks Faithfulness, Relevancy, Precision, and Recall)...


Evaluating: 100%|██████████| 16/16 [01:00<00:00,  3.81s/it]


4. Evaluation Complete! Here is your accuracy report:


,user_input,retrieved_contexts,response,reference,context_precision,context_recall,faithfulness,answer_relevancy
0,What does the CEO of Atropic say about entry-l...,[Atropic CEO saying that more than half of the...,The CEO of Atropic says that more than half of...,More than half of the entry-level jobs will di...,1.000000,1.0,1.000000,0.978129
1,What is the main focus of the podcast discusse...,"[welcome to the Indian business podcast. Kdar,...",The main focus of the podcast is to explore th...,Teaching how to build and use AI agents for la...,0.916667,1.0,0.857143,0.958660
2,What are the three vectors on which an AI syst...,"[fixed output. AI agent, it's like a coach. Yo...",The three vectors on which an AI system is bui...,"Cost, time, and quality.",1.000000,1.0,0.000000,0.999998
3,What is the difference between automation and ...,[to help me understand the gravity of the situ...,The difference between automation and AI agent...,Automation is like a vending machine with fixe...,0.583333,1.0,1.000000,0.985100



Overall Pipeline Accuracy Score: 0.89 out of 1.0
